# Task 3: RAG Core Logic and Evaluation

This notebook evaluates the RAG pipeline implementation.

In [1]:
import sys
import os
import pandas as pd
sys.path.append(os.path.abspath('..'))

from src.rag.pipeline import RAGSystem

2026-01-08 10:52:00.893457: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-08 10:52:01.009771: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/karanos/.local/lib/python3.12/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/home/karanos/.local

In [2]:
# Initialize RAG System
vector_store_path = "../vector_store/faiss_index"
rag_system = RAGSystem(vector_store_path)

Loading Vector Store from ../vector_store/faiss_index...
Vector store loaded from ../vector_store/faiss_index
  Total vectors: 40723
Loading LLM: google/flan-t5-small...


Device set to use cpu


In [3]:
# Define Test Queries
queries = [
    "Why was my loan denied?",
    "How to dispute a charge?",
    "What are the unauthorized transaction fees?",
    "My credit score dropped unexpectedly",
    "How long does a money transfer take?"
]

In [4]:
# Run Evaluation
results = []

for q in queries:
    print(f"Querying: {q}")
    try:
        response = rag_system.query(q)
        
        # Extract top snippet
        top_snippet = response['context'][0]['text'][:200] + "..." if response['context'] else "No context found"
        
        results.append({
            "Question": q,
            "Generated Answer": response['answer'],
            "Retrieved Context (Snippet)": top_snippet,
            "Quality Score (1-5)": "3",  # Placeholder
            "Comments": "Auto-generated"
        })
    except Exception as e:
        print(f"Error processing query '{q}': {e}")
        results.append({
            "Question": q,
            "Generated Answer": f"Error: {e}",
            "Retrieved Context (Snippet)": "",
            "Quality Score (1-5)": "0",
            "Comments": "Failed"
        })

df_results = pd.DataFrame(results)

Querying: Why was my loan denied?
Querying: How to dispute a charge?
Querying: What are the unauthorized transaction fees?
Querying: My credit score dropped unexpectedly
Querying: How long does a money transfer take?


In [5]:
# Display Results
from IPython.display import display
display(df_results)

,Question,Generated Answer,Retrieved Context (Snippet),Quality Score (1-5),Comments
0,Why was my loan denied?,Because of lack of credit history,they denied me because of lack of credit histo...,3,Auto-generated
1,How to dispute a charge?,"disputed the charge with the credit card, rece...",I am writing to formally dispute a charge on m...,3,Auto-generated
2,What are the unauthorized transaction fees?,"unauthorized or fraudulent charges, and fees o...",. The total amount of these charges exceeds {$...,3,Auto-generated
3,My credit score dropped unexpectedly,XXXX,"Due to the closure of the account, my oldest a...",3,Auto-generated
4,How long does a money transfer take?,My transfer was approved and processed XX/XX/X...,. Bank of America according to XXXX in the Fra...,3,Auto-generated


In [6]:
# Save Report
report_path = "../report/rag_evaluation.md"
os.makedirs(os.path.dirname(report_path), exist_ok=True)

with open(report_path, "w") as f:
    f.write("# RAG Evaluation Report\n\n")
    f.write(df_results.to_markdown(index=False))
print(f"Report saved to {report_path}")

Report saved to ../report/rag_evaluation.md
